# 5-Modul: CNN (Convolutional Neural Networks) Amaliy Loyihasi
## Ko'krak qafasi rentgen tasvirlari orqali pnevmoniyani (o'pka yallig'lanishini) aniqlash

**Loyiha maqsadi:** Ushbu amaliy loyihada zamonaviy Convolutional Neural Network (CNN) modeli loyihalashtiriladi va o'qitiladi.
Model ixtiyoriy rentgen tasviri taqdim etilganda:
- **"Sog'lom"** (NORMAL) yoki
- **"O'pka yallig'lanishi aniqlandi"** (PNEUMONIA)
ekanligini aniq ishonchlilik darajasi (confidence score) bilan birga chiqaradi.


In [ ]:
import os
import copy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Grafiklar notebook ichida to'g'ridan-to'g'ri ko'rinishi uchun
%matplotlib inline

# Hisoblash qurilmasini aniqlash (GPU yoki CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Ishlatilayotgan qurilma: {device}")


### 1. Ma'lumotlar to'plami bilan tanishish va vizualizatsiya
Ma'lumotlar to'plami `train` (80%) va `test` (20%) papkalariga ajratilgan:
- **train/**: 145 ta rentgen rasmi (72 ta Normal, 73 ta Pneumonia)
- **test/**: 38 ta rentgen rasmi (19 ta Normal, 19 ta Pneumonia)


In [ ]:
# Papkalardagi rasmlar sonini tekshiramiz
for split in ['train', 'test']:
    print(f"=== {split.upper()} to'plami ===")
    for cls in ['NORMAL', 'PNEUMONIA']:
        p = os.path.join(split, cls)
        print(f"  {cls}: {len(os.listdir(p))} ta rasm")

# Har bir sinfdan namunaviy rentgen rasmlarini ekranga chiqaramiz
def show_sample_images():
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    
    normal_sample = os.path.join('train/NORMAL', os.listdir('train/NORMAL')[0])
    pneumonia_sample = os.path.join('train/PNEUMONIA', os.listdir('train/PNEUMONIA')[0])
    
    img_norm = Image.open(normal_sample)
    img_pneu = Image.open(pneumonia_sample)
    
    axes[0].imshow(img_norm, cmap='gray')
    axes[0].set_title("Sog'lom (NORMAL)", fontsize=14, color='green', weight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(img_pneu, cmap='gray')
    axes[1].set_title("O'pka yallig'lanishi (PNEUMONIA)", fontsize=14, color='red', weight='bold')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

show_sample_images()


### 2. Ma'lumotlarni qayta ishlash va Augmentatsiya (Data Augmentation)
Rentgen tasvirlari turli o'lchamlarda bo'lgani sababli, barchasini 128x128 o'lchamga keltiramiz. Modelning umumlashtirish qobiliyatini oshirish va ortiqcha moslashish (overfitting)ning oldini olish uchun train to'plamiga tasodifiy burish (`RandomRotation`), akslantirish (`RandomHorizontalFlip`) va yorug'lik o'zgarishi (`ColorJitter`) augmentatsiyalari qo'llaniladi.


In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Lambda(lambda img: img.convert('RGB')),
        transforms.Resize((128, 128)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Lambda(lambda img: img.convert('RGB')),
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

train_dataset = datasets.ImageFolder('train', transform=data_transforms['train'])
test_dataset = datasets.ImageFolder('test', transform=data_transforms['test'])

class_names = train_dataset.classes
print(f"Sinflar: {class_names}")
print(f"Train to'plami: {len(train_dataset)} ta tasvir")
print(f"Test to'plami: {len(test_dataset)} ta tasvir")

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)


### 3. CNN (Convolutional Neural Network) Modelini Loyihalash
CNN arxitekturamiz quyidagi qismlardan iborat:
1. **4 ta Konvolyutsion blok (Feature Extractor):**
   - Har bir blokda `Conv2d` (filtrlar: 32 -> 64 -> 128 -> 256), `BatchNorm2d` (o'qish jarayonini barqarorlashtirish), `ReLU` (faollashtirish) va `MaxPool2d` (o'lchamni 2 barobar siqish) joylashgan.
2. **Klassifikator (Fully Connected Classifier):**
   - `AdaptiveAvgPool2d((2, 2))` — o'lchamni 2x2 shaklga keltiradi.
   - `Dropout` qatlamlari — neyronlarning ortiqcha moslashib qolishini oldini oladi.
   - `Linear` qatlamlar — olingan xususiyatlar asosida yakuniy 2 ta sinfga (`NORMAL` yoki `PNEUMONIA`) ehtimollik hisoblaydi.


In [ ]:
class PneumoniaCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(PneumoniaCNN, self).__init__()
        
        self.features = nn.Sequential(
            # 1-blok: 128x128 -> 64x64
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # 2-blok: 64x64 -> 32x32
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # 3-blok: 32x32 -> 16x16
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            # 4-blok: 16x16 -> 8x8
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        
        # Klassifikator (To'liq bog'langan qatlamlar)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((2, 2)),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256 * 2 * 2, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        return self.classifier(self.features(x))

model = PneumoniaCNN(num_classes=2).to(device)
print(model)


### 4. Modelni O'qitish (Training) va Eng Yaxshi Modelni Saqlash
Yo'qotish funksiyasi sifatida `CrossEntropyLoss`, optimizatsiya uchun `Adam` tanlangan.
Har bir epochda Test to'plamida model tekshiriladi va eng yuqori Test aniqligiga erishilgan model holati saqlanadi.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=6, gamma=0.5)

epochs = 15
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

best_acc = 0.0
best_model_weights = copy.deepcopy(model.state_dict())

print("CNN modelini o'qitish boshlandi...")
for epoch in range(epochs):
    # Train fazasi
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data).item()
        total += labels.size(0)
        
    scheduler.step()
    train_loss = running_loss / total
    train_acc = correct / total
    
    # Test fazasi
    model.eval()
    t_running_loss, t_correct, t_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            t_running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            t_correct += torch.sum(preds == labels.data).item()
            t_total += labels.size(0)
            
    test_loss = t_running_loss / t_total
    test_acc = t_correct / t_total
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    
    if test_acc > best_acc:
        best_acc = test_acc
        best_model_weights = copy.deepcopy(model.state_dict())
        
    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.1f}% | Test Loss: {test_loss:.4f}, Test Acc: {test_acc*100:.1f}% (Eng yaxshi: {best_acc*100:.1f}%)")

# Eng yaxshi model vaznlarini yuklash va saqlash
model.load_state_dict(best_model_weights)
torch.save(model.state_dict(), 'pneumonia_cnn.pth')
print(f"\nO'qitish muvaffaqiyatli yakunlandi! Eng yuqori Test aniqligi: {best_acc*100:.2f}%")
print("Model vaznlari 'pneumonia_cnn.pth' fayliga saqlandi.")


### 5. O'qitish Natijalarining Grafiklari (Loss va Accuracy)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss grafigi
axes[0].plot(range(1, epochs + 1), history['train_loss'], label='Train Loss', marker='o', color='#2b5c8f')
axes[0].plot(range(1, epochs + 1), history['test_loss'], label='Test Loss', marker='s', color='#d9534f')
axes[0].set_title("Yo'qotish Funksiyasi (Loss)", fontsize=14, weight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy grafigi
axes[1].plot(range(1, epochs + 1), [a * 100 for a in history['train_acc']], label='Train Accuracy', marker='o', color='#2b5c8f')
axes[1].plot(range(1, epochs + 1), [a * 100 for a in history['test_acc']], label='Test Accuracy', marker='s', color='#28a745')
axes[1].set_title("Aniqlik Ko'rsatkichi (Accuracy %)", fontsize=14, weight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Aniqlik (%)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()


### 6. Yangi Rentgen Tasvirini Bashorat Qilish Funksiyasi (`predict_xray`)
Ushbu funksiya istalgan rentgen tasviri yo'lini qabul qiladi va CNN modeli yordamida:
- **"Sog'lom"** (yashil rangda) yoki
- **"O'pka yallig'lanishi aniqlandi"** (qizil rangda)
deb aniq xulosani ishonchlilik darajasi (confidence score) bilan birga qaytaradi hamda rasmni vizual tarzda ko'rsatadi.


In [ ]:
infer_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert('RGB')),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def predict_xray(image_path, show_image=True):
    """
    Rentgen tasvirini tahlil qilib, 'Sog'lom' yoki 'O'pka yallig'lanishi aniqlandi' natijasini chiqaradi.
    """
    if not os.path.exists(image_path):
        print(f"Xatolik: '{image_path}' fayli topilmadi!")
        return None
    
    # Rasmni ochamiz
    raw_img = Image.open(image_path)
    
    # Transformatsiya qo'llaymiz
    input_tensor = infer_transform(raw_img).unsqueeze(0).to(device)
    
    # Model bashorati
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        probabilities = torch.softmax(output, dim=1)[0]
        pred_idx = torch.argmax(probabilities).item()
        confidence = probabilities[pred_idx].item() * 100
        
        prob_normal = probabilities[0].item() * 100
        prob_pneumonia = probabilities[1].item() * 100
        
    # O'zbek tilidagi rasmiy xulosa
    if pred_idx == 0:
        result_text = "Sog'lom"
        result_color = "green"
    else:
        result_text = "O'pka yallig'lanishi aniqlandi"
        result_color = "red"
        
    print("=" * 60)
    print(f" Fayl: {os.path.basename(image_path)}")
    print(f" NATIJA: {result_text}")
    print(f" Ishonchlilik darajasi: {confidence:.2f}%")
    print(f" Ehtimolliklar: Sog'lom: {prob_normal:.1f}% | O'pka yallig'lanishi: {prob_pneumonia:.1f}%")
    print("=" * 60)
    
    if show_image:
        plt.figure(figsize=(6, 6))
        plt.imshow(raw_img, cmap='gray')
        plt.title(f"{result_text}\n(Ishonchlilik: {confidence:.1f}%)", 
                  fontsize=14, color=result_color, weight='bold', pad=12)
        plt.axis('off')
        plt.show()
        
    return {
        'result': result_text,
        'confidence': confidence,
        'prob_normal': prob_normal,
        'prob_pneumonia': prob_pneumonia
    }


### 7. Sinov: Rentgen tasvirlarida amaliy tekshirib ko'rish


In [ ]:
# 1-sinov: Sog'lom rentgen tasviri
print(">>> 1-TEST: SOG'LOM RENTGEN TASVIRI <<<")
predict_xray('test/NORMAL/NORMAL2-IM-0052-0001.jpeg')

# 2-sinov: O'pka yallig'lanishi (COVID-19 / Pnevmoniya) aniqlangan tasvir
print("\n>>> 2-TEST: O'PKA YALLIG'LANISHI (PNEUMONIA) TASVIRI <<<")
predict_xray('test/PNEUMONIA/ryct.2020200034.fig2.jpeg')
